In [ ]:
import xarray as xr
import os
import numpy as np

In [ ]:
files = [f"/work/bg1186/g260200/RUN04b_JUWELS_wpile/outputs_all/out2d_{day}.nc" for day in range(1, 366)]

In [ ]:
variables = ["windSpeedX", "windSpeedY", "elevation", "depthAverageVelX", "depthAverageVelY", "sigWaveHeight", "peakPeriod", "meanWavePeriod", "TM10",
             "meanWaveNumber", "meanWaveLength", "meanWaveDirection", "meanDirSpreading",
             "SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes", "dryFlagNode", 
             "SCHISM_hgrid_node_x", "SCHISM_hgrid_node_y", "SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y",
            ]

In [ ]:
ds = xr.open_mfdataset(files, chunks="auto")

/tmp/ipykernel_395540/1482977654.py:1: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, chunks="auto")


In [ ]:
ds

<xarray.Dataset> Size: 1TB
Dimensions:                  (time: 8760, one: 1, nSCHISM_hgrid_node: 589428,
                              nSCHISM_hgrid_face: 1168634,
                              nMaxSCHISM_hgrid_face_nodes: 4,
                              nSCHISM_hgrid_edge: 1758191, two: 2)
Coordinates:
  * time                     (time) datetime64[ns] 70kB 2020-01-01T01:00:00 ....
    SCHISM_hgrid_node_x      (nSCHISM_hgrid_node) float64 5MB dask.array<chunksize=(589428,), meta=np.ndarray>
    SCHISM_hgrid_node_y      (nSCHISM_hgrid_node) float64 5MB dask.array<chunksize=(589428,), meta=np.ndarray>
    SCHISM_hgrid_face_x      (nSCHISM_hgrid_face) float64 9MB dask.array<chunksize=(1168634,), meta=np.ndarray>
    SCHISM_hgrid_face_y      (nSCHISM_hgrid_face) float64 9MB dask.array<chunksize=(1168634,), meta=np.ndarray>
    SCHISM_hgrid_edge_x      (nSCHISM_hgrid_edge) float64 14MB dask.array<chunksize=(1758191,), meta=np.ndarray>
    SCHISM_hgrid_edge_y      (nSCHISM_hgrid_edge) float64 14MB dask.array<chunksize=(1758191,), meta=np.ndarray>
Dimensions without coordinates: one, nSCHISM_hgrid_node, nSCHISM_hgrid_face,
                                nMaxSCHISM_hgrid_face_nodes,
                                nSCHISM_hgrid_edge, two
Data variables: (12/36)
    minimum_depth            (time, one) float64 70kB dask.array<chunksize=(24, 1), meta=np.ndarray>
    SCHISM_hgrid             (time, one) |S1 9kB dask.array<chunksize=(24, 1), meta=np.ndarray>
    depth                    (time, nSCHISM_hgrid_node) float32 21GB dask.array<chunksize=(24, 589428), meta=np.ndarray>
    bottom_index_node        (time, nSCHISM_hgrid_node) int32 21GB dask.array<chunksize=(24, 589428), meta=np.ndarray>
    SCHISM_hgrid_face_nodes  (time, nSCHISM_hgrid_face, nMaxSCHISM_hgrid_face_nodes) float64 328GB dask.array<chunksize=(24, 1168634, 4), meta=np.ndarray>
    SCHISM_hgrid_edge_nodes  (time, nSCHISM_hgrid_edge, two) float64 246GB dask.array<chunksize=(24, 1758191, 2), meta=np.ndarray>
    ...                       ...
    meanWaveLength           (time, nSCHISM_hgrid_node) float32 21GB dask.array<chunksize=(24, 589428), meta=np.ndarray>
    meanWaveDirection        (time, nSCHISM_hgrid_node) float32 21GB dask.array<chunksize=(24, 589428), meta=np.ndarray>
    meanDirSpreading         (time, nSCHISM_hgrid_node) float32 21GB dask.array<chunksize=(24, 589428), meta=np.ndarray>
    peakPeriod               (time, nSCHISM_hgrid_node) float32 21GB dask.array<chunksize=(24, 589428), meta=np.ndarray>
    dryFlagElement           (time, nSCHISM_hgrid_face) float32 41GB dask.array<chunksize=(24, 1168634), meta=np.ndarray>
    dryFlagSide              (time, nSCHISM_hgrid_edge) float32 62GB dask.array<chunksize=(19, 1758191), meta=np.ndarray>

In [ ]:
ds = ds[variables]
for var in variables:
    ds[var] = ds[var].astype("float32")

for coord in ds.coords:
    if coord != "time":
        ds[coord] = ds[coord].astype("float32")

In [ ]:
encoding = {var: {'chunks': (24, 589428)} for var in ds.data_vars}

In [ ]:
%%time
ds.to_zarr("/work/ka1176/shared_data/2025-wwm-emulator-schism/processed_RUN04b_JUWELS_wpile.zarr", mode="w", encoding=encoding)

/work/ka1176/caroline/gitlab/wave-wind-model-emulator/.venv/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


CPU times: user 1h 31min 55s, sys: 12min 9s, total: 1h 44min 4s
Wall time: 1h 25min 48s


In [ ]:
zarrds = xr.open_dataset("/work/ka1176/shared_data/2025-wwm-emulator-schism/processed_RUN04b_JUWELS_wpile.zarr")

In [ ]:
for var in variables:
    print(var, ":")
    print("  mean:", zarrds[var].isel(time=slice(0, 24*90)).mean().values)
    print("  std:", zarrds[var].isel(time=slice(0, 24*90)).std().values)

windSpeedX :
  mean: 5.038548
  std: 6.027648
windSpeedY :
  mean: 3.3067167
  std: 6.2532287
elevation :
  mean: 0.30683705
  std: 0.91909295
depthAverageVelX :
  mean: -0.017730717
  std: 0.30434707
depthAverageVelY :
  mean: 0.008856153
  std: 0.2007033
sigWaveHeight :
  mean: 1.7700078
  std: 1.115033
peakPeriod :
  mean: 6.191719
  std: 2.0858357
meanWavePeriod :
  mean: 5.330238
  std: 1.5788294
TM10 :
  mean: 5.254476
  std: 1.6329303
meanWaveNumber :
  mean: 0.608012
  std: 1.4502757
meanWaveLength :
  mean: 24.917128
  std: 14.2197275
meanWaveDirection :
  mean: 236.46817
  std: 75.93169
meanDirSpreading :
  mean: 33.470478
  std: 10.477335
SCHISM_hgrid_face_nodes :
  mean: 295120.34
  std: 169453.03
SCHISM_hgrid_edge_nodes :
  mean: 294983.38
  std: 169690.08
dryFlagNode :
  mean: 0.024593802
  std: 0.15488365
SCHISM_hgrid_node_x :


ValueError: Dimensions {'time'} do not exist. Expected one or more of ('nSCHISM_hgrid_node',)

In [ ]:
variables = ["windSpeedX", "windSpeedY", "sigWaveHeight","meanWavePeriod",
            "SCHISM_hgrid_edge_nodes",  
             "SCHISM_hgrid_node_x", "SCHISM_hgrid_node_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y",
            ]

In [ ]:
cicd = zarrds.isel(time=slice(10*24, 15*24), nSCHISM_hgrid_node=slice(None, None, 100000))[variables]
cicd.to_zarr("/work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr", mode="w")

/work/ka1176/caroline/gitlab/wave-wind-model-emulator/.venv/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [ ]:
!zip -r cicd.zarr.zip /work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr

  adding: work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr/ (stored 0%)
  adding: work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr/meanWavePeriod/ (stored 0%)
  adding: work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr/meanWavePeriod/c/ (stored 0%)
  adding: work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr/meanWavePeriod/c/1/ (stored 0%)
  adding: work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr/meanWavePeriod/c/1/0 (deflated 81%)
  adding: work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr/meanWavePeriod/c/3/ (stored 0%)
  adding: work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr/meanWavePeriod/c/3/0 (deflated 81%)
  adding: work/ka1176/caroline/gitlab/wave-wind-model-emulator/tests/cicd_data/cicd.zarr/meanWavePeriod/c/4/ (stored 0%)
  adding: work/ka1176/caroline/gitlab/wave-wind-m

In [ ]:
cicd

<xarray.Dataset> Size: 4GB
Dimensions:                  (time: 120, nSCHISM_hgrid_node: 6,
                              nSCHISM_hgrid_edge: 1758191, two: 2,
                              nSCHISM_hgrid_face: 1168634,
                              nMaxSCHISM_hgrid_face_nodes: 4)
Coordinates:
  * time                     (time) datetime64[ns] 960B 2020-01-11T01:00:00 ....
    SCHISM_hgrid_edge_x      (nSCHISM_hgrid_edge) float32 7MB ...
    SCHISM_hgrid_edge_y      (nSCHISM_hgrid_edge) float32 7MB ...
    SCHISM_hgrid_face_x      (nSCHISM_hgrid_face) float32 5MB ...
    SCHISM_hgrid_face_y      (nSCHISM_hgrid_face) float32 5MB ...
    SCHISM_hgrid_node_x      (nSCHISM_hgrid_node) float32 24B ...
    SCHISM_hgrid_node_y      (nSCHISM_hgrid_node) float32 24B ...
Dimensions without coordinates: nSCHISM_hgrid_node, nSCHISM_hgrid_edge, two,
                                nSCHISM_hgrid_face, nMaxSCHISM_hgrid_face_nodes
Data variables: (12/16)
    depthAverageVelX         (time, nSCHISM_hgrid_node) float32 3kB ...
    depthAverageVelY         (time, nSCHISM_hgrid_node) float32 3kB ...
    dryFlagNode              (time, nSCHISM_hgrid_node) float32 3kB ...
    elevation                (time, nSCHISM_hgrid_node) float32 3kB ...
    meanDirSpreading         (time, nSCHISM_hgrid_node) float32 3kB ...
    meanWaveDirection        (time, nSCHISM_hgrid_node) float32 3kB ...
    ...                       ...
    SCHISM_hgrid_edge_nodes  (time, nSCHISM_hgrid_edge, two) float32 2GB ...
    SCHISM_hgrid_face_nodes  (time, nSCHISM_hgrid_face, nMaxSCHISM_hgrid_face_nodes) float32 2GB ...
    sigWaveHeight            (time, nSCHISM_hgrid_node) float32 3kB ...
    TM10                     (time, nSCHISM_hgrid_node) float32 3kB ...
    windSpeedX               (time, nSCHISM_hgrid_node) float32 3kB ...
    windSpeedY               (time, nSCHISM_hgrid_node) float32 3kB ...